### Bezrealitky scrape

In [18]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

headers = {'User-Agent': 'JEM207 StudentProject (Educational use; contact: your_email@fsv.cuni.cz)'}

url = 'https://www.bezrealitky.cz/vyhledat?currency=CZK&estateType=BYT&offerType=PRONAJEM&osm_value=Praha%2C+%C4%8Cesko&regionOsmIds=R435541&location=exact'

response = requests.get(url, headers=headers)
print(response.status_code)

200


In [2]:
soup = BeautifulSoup(response.text, 'html.parser')

links = soup.find_all('a', href=True)
uris = [l['href'] for l in links if 'nabidka-pronajem-bytu' in l['href']]
uris = list(set(uris))  # remove duplicates
print(len(uris))
print(uris[:5])

15
['https://www.bezrealitky.cz/nemovitosti-byty-domy/580214-nabidka-pronajem-bytu-kratky-lan-praha-6', 'https://www.bezrealitky.cz/nemovitosti-byty-domy/1003608-nabidka-pronajem-bytu-ondrickova-praha', 'https://www.bezrealitky.cz/nemovitosti-byty-domy/1013292-nabidka-pronajem-bytu-hodkovicka-praha', 'https://www.bezrealitky.cz/nemovitosti-byty-domy/883916-nabidka-pronajem-bytu-nuslova-praha', 'https://www.bezrealitky.cz/nemovitosti-byty-domy/1013293-nabidka-pronajem-bytu-na-domovine-praha']


In [3]:
url_page2 = 'https://www.bezrealitky.cz/vyhledat?currency=CZK&estateType=BYT&offerType=PRONAJEM&osm_value=Praha%2C+%C4%8Cesko&regionOsmIds=R435541&location=exact&page=2'

response2 = requests.get(url_page2, headers=headers)
soup2 = BeautifulSoup(response2.text, 'html.parser')

links2 = soup2.find_all('a', href=True)
uris2 = [l['href'] for l in links2 if 'nabidka-pronajem-bytu' in l['href']]
uris2 = list(set(uris2))
print(len(uris2))
print(uris2[:3])

15
['https://www.bezrealitky.cz/nemovitosti-byty-domy/1004937-nabidka-pronajem-bytu-ceskomoravska', 'https://www.bezrealitky.cz/nemovitosti-byty-domy/990839-nabidka-pronajem-bytu-smilovskeho', 'https://www.bezrealitky.cz/nemovitosti-byty-domy/883916-nabidka-pronajem-bytu-nuslova-praha']


In [9]:
import time
base_url = 'https://www.bezrealitky.cz/vyhledat?currency=CZK&estateType=BYT&offerType=PRONAJEM&osm_value=Praha%2C+%C4%8Cesko&regionOsmIds=R435541&location=exact'
all_uris = set()

for page in range(1, 93):
    url = f'{base_url}&page={page}'
    try:
        r = requests.get(url, headers=headers)
        soup = BeautifulSoup(r.text, 'html.parser')
        links = soup.find_all('a', href=True)
        uris = [l['href'].split('/nemovitosti-byty-domy/')[1] 
                for l in links 
                if 'nabidka-pronajem-bytu' in l['href']]
        all_uris.update(uris)
    except:
        print(f'page {page} failed, skipping')
        time.sleep(5)
        continue
    time.sleep(3)

print(f'Total unique URIs: {len(all_uris)}')

Total unique URIs: 1375


In [8]:
len(all_uris)

405

In [17]:
build_hash = '872329e507a2c7c4f5c7e46814dcf979a4cd6731'

listings = []

for uri in all_uris:
    json_url = f'https://www.bezrealitky.cz/_next/data/{build_hash}/cs/nemovitosti-byty-domy/{uri}.json'
    try:
        r = requests.get(json_url, headers=headers)
        advert = r.json()['pageProps']['origAdvert']
        listings.append({
            'price': advert['price'],
            collected 1363 listings'charges': advert['charges'],
            'surface': advert['surface'],
            'disposition': advert['disposition'],
            'address': advert['address'],
            'district': advert['regionTree'][-1]['name']
        })
    except:
        continue
    time.sleep(2)

print(f'collected {len(listings)} listings')

collected 1363 listings


In [19]:
df = pd.DataFrame(listings)
df.to_csv('apartments.csv', index=False)
print(df.head())

   price  charges  surface disposition                                address  \
0  16100     3100       33   DISP_1_KK                 Federova, Praha - Kyje   
1  40621     7168       40   UNDEFINED          Starokošířská, Praha - Košíře   
2  28000     6000       80   DISP_3_KK             Amforova, Praha - Stodůlky   
3  15990        0       33   DISP_1_KK              Freyova, Praha - Vysočany   
4  24500     3000       51    DISP_2_1  náměstí Josefa Machka, Praha - Košíře   

         district  
0      Praha-Kyje  
1    Praha-Košíře  
2  Praha-Stodůlky  
3  Praha-Vysočany  
4    Praha-Košíře  


### Lidl

In [23]:
basket_terms = {
    'bread': 'kváskový chléb',
    'milk': 'mléko polotučné',
    'eggs': 'vejce',
    'butter': 'máslo',
    'chicken': 'kuřecí prsa',
    'pasta': 'těstoviny',
    'rice': 'rýže',
    'chopped_tomatoes': 'krájená rajčata',
    'onions': 'cibule',
    'frozen_vegetables': 'mražená zelenina',
    'yoghurt': 'jogurt',
    'coffee': 'instantní káva',
    'bananas': 'banány'
}

lidl_prices = {}

for item, term in basket_terms.items():
    try:
        url = f'https://www.lidl.cz/q/api/search?q={term}&locale=cs_CZ&assortment=CZ&version=2.1.0&fetchsize=20'
        r = requests.get(url, headers=headers)
        data = r.json()
        prices = []
        for product in data['items']:
            try:
                title = product['gridbox']['data']['fullTitle']
                price = product['gridbox']['data']['price']['price']
                packaging = product['gridbox']['data']['price'].get('packaging', {}).get('text', 'N/A')
                prices.append({'title': title, 'price': price, 'packaging': packaging})
            except:
                continue
        if prices:
            lidl_prices[item] = min(prices, key=lambda x: x['price'])
    except:
        continue
    time.sleep(1)

In [25]:
df_lidl = pd.DataFrame(lidl_prices).T
df_lidl.index.name = 'item'
df_lidl.to_csv('lidl_prices.csv')

,title,price,packaging
item,,,
bread,Jablečná kapsa,12.9,105 g
milk,Pšeničná mouka polohrubá,9.9,1 kg
eggs,Špagety,11.9,500 g
butter,Máslový croissant,7.9,74 g
chicken,BIO Kuřecí prsní řízky,59.9,Dostupné pouze ve vybraných prodejnách


### Rohlik.cz

In [26]:
rohlik_prices = {}

for item, term in basket_terms.items():
    try:
        url = f'https://www.rohlik.cz/services/frontend-service/search-metadata?search={term}&offset=0&limit=15&companyId=1&filterData=%7B%22filters%22%3A%5B%5D%7D&canCorrect=true'
        r = requests.get(url, headers=headers)
        products = r.json()['data']['productList']
        
        if products:
            cheapest = min(products, key=lambda x: x['price']['full'])
            rohlik_prices[item] = {
                'title': cheapest['productName'],
                'price': cheapest['price']['full'],
                'packaging': cheapest['textualAmount']
            }
    except:
        continue
    time.sleep(1)

df_rohlik = pd.DataFrame(rohlik_prices).T
df_rohlik.index.name = 'item'
df_rohlik.to_csv('rohlik_prices.csv')
print(df_rohlik)

                                                            title  price  \
item                                                                       
bread                           Breadway Kváskový toustáček žitný   37.9   
milk                          Miil Trvanlivé polotučné mléko 1,5%   14.9   
eggs                                 Varmuža Ruské vejce v aspiku   34.9   
butter                                            Dr. Halíř Máslo   40.9   
chicken                              Světničkové kuře prsní řízek  59.78   
pasta                                      Kitchin Spaghetti N. 5   15.9   
rice                                       Kitchin Rýže jasmínová   24.9   
chopped_tomatoes   Kitchin Celá loupaná rajčata v rajčatové šťávě   24.9   
onions                                          Cibule žlutá 1 ks   1.85   
frozen_vegetables               Agro Jesenice Zelenina s kukuřicí   20.9   
yoghurt                       Miil Krémový jogurt bílý 3,7 % tuku    7.9   
coffee      